# Lab 6.2 &mdash; Understanding Embeddings

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 2 &middot; Module 6 &mdash; Agentic RAG**

### What you'll do
- Turn a sentence into 384 numbers and look at them
- Score two texts by meaning with cosine similarity, and predict the answer first
- Decide which of the two <code>Embeddings</code> methods a query needs, and which a corpus needs
- Build a search engine in fifteen lines &mdash; which is all a vector store does

> **How this lab works.** You write real LangChain code. Fill every `BLANK`, then run the
> **Self-check** cell under each section &mdash; those check the *objects you built* (a
> `Document`, a Chroma collection, a retriever, a chain), so they are deterministic and do not
> depend on the chat model. Cells marked **Run it for real** put your code in front of the
> sandbox model; that is the part worth watching. The score line is feedback, not a grade.

> **Two different models are in play, and only one of them is billed.** The **chat model**
> (`qwen36-35b-a3b-lab`) answers questions and is reached over the gateway. The **embedding
> model** (`all-MiniLM-L6-v2`, 384 dimensions) turns text into vectors and runs on this pod's
> own CPU &mdash; no key, no gateway, no tokens. Keeping them straight is most of Module 6.

> **This lab makes no gateway calls at all.** Everything here runs on the pod's own
> CPU, costs nothing, and would work with the network unplugged.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, math, textwrap, warnings
from typing import Any, Callable

warnings.filterwarnings("ignore")     # sentence-transformers is chatty on first import

WORK = os.path.join("/tmp", "awmas-lab-6-02")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the CHAT model: qwen, through the sandbox gateway -------------------
# Already configured -- nothing to install, no key to register. Read from the
# environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens. Thinking is off by default here because you will make a lot of calls today;
# pass think=True to any call below to see the difference for yourself.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Chat model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

# ---- the EMBEDDING model: local, free, nothing to configure --------------
# all-MiniLM-L6-v2, 384 dimensions. It runs on this pod's CPU and has nothing to do with
# the chat model above: no gateway, no key, no tokens billed. The cache is already warm
# in your sandbox, so the first call is a second or two, not a download.
#
# It is reached through onnxruntime rather than torch, and that is a measured choice
# rather than a taste: same model, same vectors, ~170 MB of memory instead of ~840. Your
# whole sandbox has 2.5 GB for every notebook you leave open, and a kernel you have
# forgotten about is still holding its share.
from langchain_core.embeddings import Embeddings

class MiniLMEmbeddings(Embeddings):
    """all-MiniLM-L6-v2 behind LangChain's Embeddings interface.

    Two methods is the whole contract -- which is why a store, a splitter and a chain
    never need to know which model is underneath, or what runtime it uses."""

    def __init__(self):
        from chromadb.utils.embedding_functions import ONNXMiniLM_L6_V2
        self._fn = ONNXMiniLM_L6_V2()

    def embed_documents(self, texts: list) -> list:
        return [[float(x) for x in v] for v in self._fn(list(texts))]

    def embed_query(self, text: str) -> list:
        return [float(x) for x in self._fn([text])[0]]


_emb_cache = {}
def get_embeddings():
    """The embedding model, built once per kernel."""
    if "model" not in _emb_cache:
        _emb_cache["model"] = MiniLMEmbeddings()
    return _emb_cache["model"]

print("work dir   :", WORK)
print("chat model :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

## Concept

An **embedding** is a position. The model reads a piece of text and returns a point in a
384-dimensional space, chosen so that text meaning similar things lands nearby.

Two texts are then compared by the *angle* between their vectors &mdash; **cosine
similarity**, which runs from 1.0 (same direction) down through 0 (unrelated). That is
the whole mechanism. A vector store is this, plus an index so you do not have to score
every document one at a time.

`Embeddings` has exactly two methods, and the difference matters:

| | |
|---|---|
| `embed_query(text)` | one string &rarr; one vector. For the thing being searched *with* |
| `embed_documents(texts)` | a list &rarr; a list of vectors. For the things being searched *over* |

## Section 1 &mdash; One sentence, 384 numbers

The first call loads the model. It is already cached in your sandbox, so this is a second
or two rather than a download.

In [ ]:
def embed_one(text: str) -> list:
    """A single search string. Which of the two methods is that?"""
    emb = get_embeddings()
    method = emb.embed_query
    return method(text)


def cosine(a: list, b: list) -> float:
    """Given -- this is arithmetic, not a design decision."""
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))

In [ ]:
# --- Self-check: Section 1   (the embedding model runs locally -- no gateway)
check("embed_one returns a vector, not a list of vectors",
      lambda: isinstance(embed_one("annual leave")[0], float),
      "embed_documents would give you a list of lists here")
check("the vector has 384 dimensions",
      lambda: len(embed_one("annual leave")) == 384)
check("the same text always gives the same vector",
      lambda: embed_one("annual leave") == embed_one("annual leave"),
      "embeddings are deterministic -- unlike the chat model, nothing is sampled")
check("a text about something else lands somewhere else",
      lambda: cosine(embed_one("annual leave"), embed_one("annual leave")) >
              cosine(embed_one("annual leave"), embed_one("the Mumbai office")))

def _look():
    v = embed_one("What is the refund policy?")
    print("first 8 of 384:", [round(x, 4) for x in v[:8]])
guard(_look)

## Section 2 &mdash; Predict, then measure

Three phrasings of a question about getting money back, and one about the weather. Write
down which pair you expect to score highest *before* you run it &mdash; the point of the
exercise is the gap between the guess and the number.

In [ ]:
PAIRS = {
    "refund/return":  ("What is the refund policy?", "How do I return a product?"),
    "refund/weather": ("What is the refund policy?", "What is the weather in Mumbai?"),
    "refund/money":   ("What is the refund policy?", "How do I get my money back?"),
}

def most_similar_pair() -> str:
    """Which of the three keys above do you expect to score highest? No shared words
    is allowed to win -- that is the whole claim embeddings make."""
    return "refund/money"

def score_pair(name: str) -> float:
    """Given."""
    a, b = PAIRS[name]
    return cosine(embed_one(a), embed_one(b))

In [ ]:
# --- Self-check: Section 2
check("your prediction names one of the three pairs",
      lambda: most_similar_pair() in PAIRS)
check("and the measurement agrees with it",
      lambda: max(PAIRS, key=score_pair) == most_similar_pair(),
      "run the cell below, read the three numbers, and change your answer")
check("the unrelated pair scores lowest",
      lambda: min(PAIRS, key=score_pair) == "refund/weather")

def _table():
    for name in PAIRS:
        a, b = PAIRS[name]
        print(f"  {score_pair(name):.4f}  {name:16} {a!r} vs {b!r}")
guard(_table)

In [ ]:
def embed_corpus(texts: list) -> list:
    """Many documents at once. Which method, and why is it not the same one?"""
    emb = get_embeddings()
    method = emb.embed_documents
    return method(texts)


def rank(query: str, texts: list, vectors: list) -> list:
    """Score every document against the query, best first. Given -- a sort is a sort."""
    qv = embed_one(query)
    return sorted(((cosine(qv, v), t) for t, v in zip(texts, vectors)), reverse=True)

In [ ]:
# --- Self-check: the fifteen-line search engine
DOCS = [
    "Refund within 30 days of purchase",
    "Free shipping on orders above 500",
    "Contact support on the internal helpdesk",
    "Return items in their original packaging",
]

def _vecs():
    return embed_corpus(DOCS)

check("embed_corpus returns one vector per document",
      lambda: len(_vecs()) == len(DOCS))
check("each of them is 384 long",
      lambda: all(len(v) == 384 for v in _vecs()))
check("'how do I get my money back' finds the refund line",
      lambda: rank("How do I get my money back?", DOCS, _vecs())[0][1].startswith("Refund"),
      "not one word of the query appears in that document")
check("and the shipping line is not the top hit for it",
      lambda: "shipping" not in rank("How do I get my money back?", DOCS, _vecs())[0][1])

def _search():
    for s, t in rank("How do I get my money back?", DOCS, _vecs()):
        print(f"  [{s:.4f}] {t}")
guard(_search)

In [ ]:
score()

## Your turn

1. Add `"Refunds are processed within 5-7 business days"` to `DOCS` and search again. Two
   documents now deserve to be returned. Does the ranking put them 1 and 2? This is the
   first hint of why `k` is a decision and not a constant.
2. Score `"Python is a programming language"` against `"Python is a snake"`. The number is
   higher than you want it to be. Embeddings capture *topic* at least as much as meaning,
   and that is a failure mode you will meet in production.
3. Swap `cosine` for a plain dot product and re-run Section 2. The ranking barely moves
   here &mdash; because these vectors are already close to unit length. On a corpus of very
   uneven document lengths it moves a lot.